# Lekcja 05 - Agentic RAG


## Konfiguracja

Ten notatnik demonstruje wzorzec Agentic RAG (Retrieval-Augmented Generation) z użyciem Microsoft Agent Framework.

**Wymagania wstępne:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — punkt końcowy twojej usługi Azure AI Search
- `AZURE_SEARCH_API_KEY` — klucz API twojej usługi Azure AI Search
- wdrożenie Azure OpenAI skonfigurowane za pomocą zmiennych środowiskowych
- uwierzytelnienie w Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Czym jest Agentic RAG?

Tradycyjny RAG podąża za stałym procesem: najpierw pobieranie dokumentów, a następnie generowanie odpowiedzi. **Agentic RAG** idzie dalej, dając agentowi autonomię w decydowaniu **kiedy** i **jak** pobierać informacje.

Dzięki Agentic RAG agent może:
- **Zdecydować**, czy pobieranie informacji jest potrzebne przed udzieleniem odpowiedzi na pytanie
- **Wybrać**, które źródło danych lub narzędzie zapytać
- **Ocenić** pozyskane wyniki i wykonać dalsze pobieranie, jeśli pierwsza próba jest niewystarczająca
- **Połączyć** informacje z wielu etapów pobierania w spójną odpowiedź

To sprawia, że agent jest bardziej elastyczny i dokładny w porównaniu do statycznego procesu pobierz-then-generuj.


## Tworzenie narzędzia do wyszukiwania

W Agentic RAG zewnętrzne źródła danych są opakowane jako **narzędzia**, które agent może wywołać na żądanie. Pozwala to agentowi traktować wyszukiwanie jako kolejną akcję, którą może wykonać, a nie jako obowiązkowy krok.

Poniżej definiujemy bazę wiedzy o podróżach i udostępniamy ją jako narzędzie, które agent może wywołać, aby znaleźć informacje o destynacjach.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Tworzenie agenta RAG

Teraz tworzymy agenta, który został poinstruowany, aby **zawsze najpierw wyszukiwać informacje przed udzieleniem odpowiedzi**. Agent korzysta z narzędzia `search_travel_knowledge`, aby oprzeć swoje odpowiedzi na bazie wiedzy, zamiast polegać na własnych danych treningowych.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteracyjne wyszukiwanie — wzorzec Twórca-Kontroler

Kluczową zaletą Agentic RAG jest **iteracyjne wyszukiwanie**. Agent może przeprowadzać wielokrotne rundy poszukiwań, aby zweryfikować, dopracować lub rozszerzyć swoje początkowe ustalenia — podobnie jak w procesie "twórca-kontroler":

1. **Krok twórcy**: Agent pozyskuje początkowe informacje i sporządza odpowiedź.
2. **Krok kontrolera**: Agent wykonuje dodatkowe wyszukiwania, aby zweryfikować szczegóły lub uzupełnić luki.

Poniżej agentowi zadano pytanie, które wymaga porównania kilku miejsc docelowych, co skłania go do wielokrotnego wyszukiwania.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Podsumowanie

W tej lekcji nauczyłeś się, jak zbudować system **Agentic RAG** z wykorzystaniem Microsoft Agent Framework:

- **Agentic RAG** pozwala agentom autonomicznie decydować, kiedy pobierać informacje, czyniąc pobieranie dynamicznym, a nie stałym.
- **Narzędzia jako źródła danych**: Zewnętrzne bazy wiedzy (takie jak Azure AI Search) są opakowane jako narzędzia, które agent może wywoływać.
- **Iteracyjne pobieranie**: Wzorzec maker-checker pozwala agentowi wykonać wiele rund pobierania — wyszukiwanie, weryfikację i doprecyzowanie — zanim wygeneruje ostateczną odpowiedź.

W środowisku produkcyjnym należy zastąpić pamięciową bazę `TRAVEL_KNOWLEDGE_BASE` rzeczywistym indeksem Azure AI Search, aby obsługiwać pobieranie dokumentów podróżnych na dużą skalę.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony z wykorzystaniem usługi tłumaczeń AI [Co-op Translator](https://github.com/Azure/co-op-translator). Chociaż dokładamy starań, aby tłumaczenie było jak najbardziej precyzyjne, należy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym powinien być uznawany za wiarygodne źródło informacji. W przypadku istotnych informacji zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonywanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
